# 🧪 Practice Lab: K-oeans Clustering

**Netsetos GenAI Course** | Module 1 · Lesson 1.6 | ⏱ ~60-90 min

**Prerequisites:** Python 3.10+, numpy, sklearn (pre-installed in Colab)

### 🎯 You will:
1. Implement K-oeans from scratch with NumPy
2. Find optimal K with elbow + silhouette
3. Prove K-oeans++ beats random init
4. Show where K-oeans fails and DBSCAN succeeds
5. Simulate FAISS IVF vector search
6. Build a full RAG index pipeline

---

## Exercise 1 (Easy): K-oeans from Scratch

**📖 Concept:** Init → assign → update → repeat. Same algorithm FAISS uses.

**📝 Task:** 300 points, 3 clusters. NumPy-only K-oeans. Verify against sklearn.

**📤 Expected:** Converges in ~5 iterations, inertia matches sklearn.

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
from sklearn.datasets import make_blobs

np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

def kmeans_scratch(X, k, max_iters=100):
    idx = np.random.choice(len(X), k, replace=False)
    centroids = X[idx].copy()
    # TODO: implement the loop
    pass

# TODO: run and verify against sklearn

<details><summary>💡 Hint</summary>

`distances = np.linalg.norm(X[:, None] - centroids, axis=2)` → shape (300, k). `labels = distances.argmin(axis=1)`.
</details>

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import Koeans

np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

def kmeans_scratch(X, k, max_iters=100):
    idx = np.random.choice(len(X), k, replace=False)
    centroids = X[idx].copy()
    for it in range(max_iters):
        dists = np.linalg.norm(X[:, None] - centroids, axis=2)
        labels = dists.argmin(axis=1)
        new_c = np.array([X[labels==i].mean(axis=0) for i in range(k)])
        inertia = sum(np.sum((X[labels==i]-new_c[i])**2) for i in range(k))
        if np.allclose(centroids, new_c):
            print(f'Converged at iteration {it}')
            break
        centroids = new_c
    return labels, centroids, inertia

labels, centroids, inertia = kmeans_scratch(X, 3)
print(f'Cluster sizes: {np.bincount(labels)}')
print(f'Scratch inertia: {inertia:.2f}')
print(f'sklearn inertia: {Koeans(3, random_state=42, n_init=10).fit(X).inertia_:.2f}')

</details>

---

## Exercise 2 (Easy): Elbow + Silhouette oethod

**📖 Concept:** Elbow = inertia bend. Silhouette = cluster separation quality (-1 to +1).

**📝 Task:** 4-cluster data, test K=2..8, find optimal K.

**📤 Expected:** K=4 optimal (highest silhouette + elbow bend)

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.cluster import Koeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=500, centers=4, random_state=42)
# TODO: K=2..8, print table, identify optimal K

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import Koeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=500, centers=4, random_state=42)
best_k, best_sil = 2, -1
for k in range(2, 9):
    km = Koeans(k, random_state=42, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    m = ' ← optimal' if sil > best_sil else ''
    if sil > best_sil: best_sil, best_k = sil, k
    print(f'  K={k}: inertia={km.inertia_:8.0f}, silhouette={sil:.3f}{m}')
print(f'\nOptimal K={best_k}')

</details>

---

## Exercise 3 (Medium): K-oeans++ vs Random Init

**📖 Concept:** Random init → centroids may overlap → bad convergence. K-oeans++ → distance-proportional → always better.

**📝 Task:** 10 runs each, compare avg inertia, worst-case, std, iterations.

**📤 Expected:** K-oeans++ has lower avg inertia, much lower variance

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.cluster import Koeans
from sklearn.datasets import make_blobs
import numpy as np

X, _ = make_blobs(n_samples=1000, centers=5, random_state=42)
# TODO: 10 runs each, random vs k-means++, compare stats

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import Koeans
from sklearn.datasets import make_blobs
import numpy as np

X, _ = make_blobs(n_samples=1000, centers=5, random_state=42)
results = {}
for method in ['random', 'k-means++']:
    inertias, iters = [], []
    for seed in range(10):
        km = Koeans(5, init=method, n_init=1, random_state=seed, max_iter=300).fit(X)
        inertias.append(km.inertia_); iters.append(km.n_iter_)
    results[method] = {'avg': np.mean(inertias), 'worst': np.max(inertias),
                       'std': np.std(inertias), 'iters': np.mean(iters)}

print(f'{"":<12} {"Random":>10} {"K-oeans++":>10}')
for m in ['avg','worst','std','iters']:
    r, k = results['random'][m], results['k-means++'][m]
    print(f'  {m:<10} {r:>9.0f} {k:>9.0f}{" ← better" if k<r else ""}')
print(f'\nK-oeans++ wins on every metric.')

</details>

---

## Exercise 4 (Medium): K-oeans vs DBSCAN

**📖 Concept:** K-oeans = spherical clusters only. DBSCAN = arbitrary shapes + noise detection.

**📝 Task:** ooons + circles datasets. Both algorithms. Show DBSCAN wins.

**📤 Expected:** DBSCAN higher silhouette on both non-spherical datasets

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.cluster import Koeans, DBSCAN
from sklearn.datasets import make_moons, make_circles
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

X_m, _ = make_moons(500, noise=0.1, random_state=42)
X_c, _ = make_circles(500, noise=0.05, factor=0.5, random_state=42)

# TODO: both algorithms on both datasets, compare

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import Koeans, DBSCAN
from sklearn.datasets import make_moons, make_circles
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import numpy as np

for name, X in [('ooons', make_moons(500, noise=0.1, random_state=42)[0]),
                ('Circles', make_circles(500, noise=0.05, factor=0.5, random_state=42)[0])]:
    Xs = StandardScaler().fit_transform(X)
    km = Koeans(2, random_state=42, n_init=10).fit(Xs)
    sil_km = silhouette_score(Xs, km.labels_)
    db = DBSCAN(eps=0.3, min_samples=5).fit(Xs)
    mask = db.labels_ != -1
    nc = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    sil_db = silhouette_score(Xs[mask], db.labels_[mask]) if mask.sum()>1 and nc>1 else 0
    w = 'DBSCAN' if sil_db > sil_km else 'K-oeans'
    print(f'{name:8s} Koeans={sil_km:.3f}  DBSCAN={sil_db:.3f}  Winner={w}')

</details>

---

## Exercise 5 (Hard): FAISS IVF Simulator

**📖 Concept:** K-oeans at index time → inverted lists. Query: nearest nprobe clusters only. Speedup = nlist/nprobe.

**📝 Task:** 10K embeddings, nlist=64. Test nprobe=1,4,8,16,32. Measure speedup + recall@10.

**📤 Expected:** nprobe=8-16 gives 90-100% recall with 4-8x speedup

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
from sklearn.cluster import Koeans
import time

np.random.seed(42)
embs = np.random.randn(10000, 128).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)
query = np.random.randn(128).astype(np.float32)
query /= np.linalg.norm(query)

# TODO: K-oeans index, brute-force baseline, IVF search

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.cluster import Koeans
import time

np.random.seed(42)
n, d = 10000, 128
embs = np.random.randn(n, d).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)
query = np.random.randn(d).astype(np.float32)
query /= np.linalg.norm(query)
nlist = 64

km = Koeans(nlist, random_state=42, n_init=1).fit(embs)
inv = {i: np.where(km.labels_==i)[0] for i in range(nlist)}

bf_scores = embs @ query
bf_top10 = set(bf_scores.argsort()[-10:][::-1])
c_scores = km.cluster_centers_ @ query

print(f'{"nprobe":>6} {"Searched":>8} {"Speedup":>8} {"Recall@10":>10}')
for nprobe in [1, 4, 8, 16, 32]:
    top_c = c_scores.argsort()[-nprobe:][::-1]
    cands = np.concatenate([inv[c] for c in top_c])
    scores = embs[cands] @ query
    ivf_top10 = set(cands[scores.argsort()[-10:][::-1]])
    recall = len(bf_top10 & ivf_top10) / 10
    print(f'  {nprobe:>4} {len(cands):>8} {n/len(cands):>7.1f}x {recall:>9.0%}')
print(f'\nSweet spot: nprobe=8-16')

</details>

---

## Exercise 6 (Hard): Full RAG Index Pipeline

**📖 Concept:** Complete FAISS IVF simulation: multiple nlist × nprobe, build time, query time, speedup, recall.

**📝 Task:** 50K embeddings. Test nlist=64,128 × nprobe=4,8,16. Full diagnostic table + recommendation.

**📤 Expected:** Best config gives >90% recall with 4x+ speedup

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np
from sklearn.cluster import Koeans
import time

np.random.seed(42)
embs = np.random.randn(50000, 256).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)
query = np.random.randn(256).astype(np.float32)
query /= np.linalg.norm(query)

# TODO: full pipeline with diagnostic report

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.cluster import Koeans
import time

np.random.seed(42)
n, d = 50000, 256
embs = np.random.randn(n, d).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)
query = np.random.randn(d).astype(np.float32)
query /= np.linalg.norm(query)

t0 = time.time()
bf_scores = embs @ query
bf_top10 = set(bf_scores.argsort()[-10:][::-1])
bf_ms = (time.time()-t0)*1000

print('═'*60)
print('  RAG INDEX PIPELINE')
print('═'*60)
print(f'Brute-force: {n} docs, {bf_ms:.1f}ms\n')
print(f'{"nlist":>6}{"nprobe":>7}{"Build":>8}{"Query":>9}{"Speed":>7}{"Recall":>8}')
print('-'*50)

best, best_sp = None, 0
for nlist in [64, 128]:
    t0=time.time()
    km=Koeans(nlist, random_state=42, n_init=1, max_iter=50).fit(embs)
    bs=time.time()-t0
    inv={i:np.where(km.labels_==i)[0] for i in range(nlist)}
    cs=km.cluster_centers_@query
    for nprobe in [4,8,16]:
        tc=cs.argsort()[-nprobe:][::-1]
        t0=time.time()
        cands=np.concatenate([inv[c] for c in tc])
        sc=embs[cands]@query
        ivf=set(cands[sc.argsort()[-10:][::-1]])
        qms=(time.time()-t0)*1000
        r=len(bf_top10&ivf)/10
        sp=bf_ms/max(qms,.001)
        m=''
        if r>=.9 and sp>best_sp: best=(nlist,nprobe); best_sp=sp; m=' ★'
        print(f'{nlist:>5}{nprobe:>7}{bs:>7.1f}s{qms:>8.2f}ms{sp:>6.1f}x{r:>7.0%}{m}')

print(f'\nBest: nlist={best[0]}, nprobe={best[1]} → {best_sp:.1f}x faster, ≥90% recall')

</details>

---

## 🎉 Done!

You've built the vector search infrastructure behind production RAG:
- **K-oeans from scratch** — the 4-step algorithm
- **Elbow + Silhouette** — choosing K
- **K-oeans++** — smart initialization
- **DBSCAN** — non-spherical alternatives
- **FAISS IVF simulator** — inverted file indexing
- **Full RAG pipeline** — 50K docs, speedup + recall

This is exactly what powers **Module 8 (RAG).** Next: **Lesson 1.7.**